# 28 — Going live: an LLM behind the decision boundary

Notebook 01 made a promise: *"replace the deterministic policy with an LLM decision boundary only after the tool contract and step budget are clear."* Twenty-seven notebooks later both are clear, and this notebook flips the switch.

**Configuration is a twin of the dspy-lab course, so one `.env` serves both repositories:** `cp .env.example .env`, set `OPENROUTER_API_KEY` (one key, every major model — create one at openrouter.ai → Keys), and optionally set `MODEL` / `STRONG_MODEL` to any [LiteLLM provider string](https://docs.litellm.ai/docs/providers) (`openrouter/...`, `openai/...`, `anthropic/...`, `ollama_chat/...`). Unset variables fall back to the same defaults as dspy-lab. Install the extra: `pip install -e '.[llm]'`.

The helper is **`notebooks/llm.py`**: `model()` / `ready()` / `configure()` read the environment, `complete()` makes one LiteLLM call, and `llm_policy()` adapts a model into the lab's `choose_action` contract — the model sees the goal, a tool catalog, and the recent trace, and must answer with one JSON action. Every entry point takes a `mock` argument that routes through LiteLLM's `mock_response`, exercising the full code path with **no key and no network** — which is how the tests cover this module and how the middle cell below runs.

What does *not* change is the contract around the model: the loop still owns the step budget, unknown tools still fail cleanly, an unparseable reply becomes a terminating answer, and consequential tools still deserve notebook 05's approval gates. The model is a policy, not an authority.

Primary sources: [LiteLLM providers](https://docs.litellm.ai/docs/providers), [OpenRouter](https://openrouter.ai) and its [model list](https://openrouter.ai/models).

In [ ]:
import sys
from importlib.util import find_spec
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))

LLM_AVAILABLE = find_spec('litellm') is not None
if LLM_AVAILABLE:
    import llm
    print('model:', llm.model(), '| strong:', llm.model(strong=True), '| key present:', llm.ready())
    # llm.configure()   # uncomment to be prompted for the key instead of using .env
else:
    print("Install the optional extra first: pip install -e '.[llm]'")

In [ ]:
# The full plumbing, offline: mocked replies exercise the real code path -
# prompt building, the LiteLLM call, JSON parsing, and the loop contract.
if LLM_AVAILABLE:
    from lab_core.patterns import run_tool_loop

    facts = {'react': 'ReAct interleaves reasoning traces with actions.'}
    tools = {'lookup': lambda term: facts.get(str(term).lower(), 'not found')}

    policy = llm.llm_policy(
        {'lookup': 'Look up a fact by keyword.'},
        mocks=['{"tool": "lookup", "input": "react"}',
               '{"final": "ReAct interleaves reasoning with tool use."}'],
    )
    result = run_tool_loop('Explain ReAct', policy, tools, max_steps=4)
    print('completed:', result.completed, '|', result.answer)

    # A hostile reply cannot wedge the loop: it degrades to a terminating answer.
    rogue = llm.llm_policy({'lookup': 'Look up a fact.'}, mocks=['I refuse to emit JSON'])
    print('rogue reply ->', run_tool_loop('goal', rogue, tools, max_steps=2).answer)
else:
    print('skipped: litellm not installed')

In [ ]:
# Live: the same loop with the model actually deciding. Costs a fraction of a cent.
if LLM_AVAILABLE and llm.ready():
    live_policy = llm.llm_policy(
        {'lookup': 'Look up a fact by keyword, e.g. react.'},
        system='Use the tools when they help; finish as soon as the goal is met.',
    )
    result = run_tool_loop('Explain the ReAct pattern in one sentence', live_policy, tools, max_steps=6)
    print('completed:', result.completed)
    for step in result.trace:
        print(f"  step {step['step']}: {step.get('tool', 'final')}")
    print(result.answer)
elif LLM_AVAILABLE:
    print('set OPENROUTER_API_KEY in .env (cp .env.example .env) to run the live cell')
else:
    print('skipped: litellm not installed')

**Where to take it.** The same adapter pattern upgrades every deterministic policy in the lab: a `DeepAgent` policy is `policy(goal, todos, trace, context)` — render `context` as the prompt (it already carries the system text, plan, file catalog, and any loaded skills) and the model drives Track E's plan loop; notebook 27's tracer will show you every decision it makes, with latencies. Swapping providers is editing `MODEL` in `.env` — the program does not change, which is the same portability argument the protocol and framework tracks kept making. **Exercise:** wire `llm_policy` into notebook 24's deep agent and watch a live model manage the todo artifact — then reread notebook 05 and decide which of its tools deserve an `ApprovalGate` before you let it.